In [ ]:
import xml.etree.ElementTree as ET
import json


def parse_tmx(path):
    root = ET.fromstring(open(path, encoding='utf-8').read())
    pairs = []

    for tu in root.findall('.//tu'):
        en = nb = None
        for tuv in tu.findall('tuv'):
            lang = tuv.get('{http://www.w3.org/XML/1998/namespace}lang')
            seg = tuv.find('seg')
            text = seg.text.strip() if seg is not None and seg.text else ''
            if lang == 'en' and text:
                en = text
            elif lang == 'nb' and text:
                nb = text
        if en and nb:
            pairs.append((nb, en))

    return pairs


def save_parallel(pairs, prefix):
    with open(f'{prefix}.no', 'w', encoding='utf-8') as f_no, \
         open(f'{prefix}.en', 'w', encoding='utf-8') as f_en:
        for nb, en in pairs:
            f_no.write(nb + '\n')
            f_en.write(en + '\n')
    print(f'{prefix}.no / .en  ({len(pairs)} lines)')


def save_tsv(pairs, path):
    with open(path, 'w', encoding='utf-8') as f:
        f.write('norwegian\tenglish\n')
        for nb, en in pairs:
            f.write(f'{nb}\t{en}\n')
    print(f'{path}  ({len(pairs)} pairs)')


def save_jsonl(pairs, path):
    with open(path, 'w', encoding='utf-8') as f:
        for nb, en in pairs:
            json.dump({'source': en, 'target': nb,
                       'source_lang': 'en', 'target_lang': 'no'},
                      f, ensure_ascii=False)
            f.write('\n')
    print(f'{path}  ({len(pairs)} pairs)')


def save_json(pairs, path):
    data = [
        {'source': en, 'target': nb, 'source_lang': 'en', 'target_lang': 'no'}
        for nb, en in pairs
    ]
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    print(f'{path}  ({len(data)} pairs)')


def save_hf(pairs, path):
    data = {'translation': [{'nb': nb, 'en': en} for nb, en in pairs]}
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    print(f'{path}  ({len(data)} pairs)')


_SAVERS = {
    'parallel':    lambda pairs, name: save_parallel(pairs, name),
    'tsv':         lambda pairs, name: save_tsv(pairs, f'{name}.tsv'),
    'jsonl':       lambda pairs, name: save_jsonl(pairs, f'{name}.jsonl'),
    'json':        lambda pairs, name: save_json(pairs, f'{name}.json'),
    'huggingface': lambda pairs, name: save_hf(pairs, f'{name}_hf.json'),
    'fairseq':     lambda pairs, name: save_parallel(pairs, name),
}


def convert(input_file, fmt='json', output_name=None):
    pairs = parse_tmx(input_file)
    print(f'Extracted {len(pairs)} pairs from {input_file}\n')
    for i, (nb, en) in enumerate(pairs[:3], 1):
        print(f'{i}. {en}\n   {nb}\n')

    name = output_name or input_file.replace('.tmx', '').replace('.', '_')
    saver = _SAVERS.get(fmt)
    if saver is None:
        raise ValueError(f'unknown format: {fmt}')
    saver(pairs, name)


if __name__ == '__main__':
    convert('npd.no.en-nb.tmx', 'json', 'npd_training_mt')

Extracted 26324 pairs from npd.no.en-nb.tmx

1. Well 16/1-23 S will be drilled from the Rowan Viking drilling facility at position 58°49’47.04’’ north 02°16’56.00’’ east in production licence 338.
   Brønn 16/1-23 S skal borast frå boreinnretninga Rowan Viking i posisjon 58°49’47,04’’ nord 02°16’56,00’’ aust i utvinningsløyve 338.

2. The drilling programme for well 16/1-23 S relates to the drilling of an appraisal well in production licence 338.
   Boreprogrammet for brønn 16/1-23 S gjeld boring av avgrensingsbrønn i utvinningsløyve 338.

3. Lundin is the operator with a 50 per cent ownership interest.
   Lundin er operatør med 50 prosent del.

npd_no_en.json  (26324 pairs)


In [ ]:
import xml.etree.ElementTree as ET
import json


def parse_tmx(path):
    root = ET.fromstring(open(path, encoding='utf-8').read())
    pairs = []

    for tu in root.findall('.//tu'):
        en = nb = None
        for tuv in tu.findall('tuv'):
            lang = tuv.get('{http://www.w3.org/XML/1998/namespace}lang')
            seg = tuv.find('seg')
            text = seg.text.strip() if seg is not None and seg.text else ''
            if lang == 'en' and text:
                en = text
            elif lang == 'nb' and text:
                nb = text
        if en and nb:
            pairs.append((nb, en))

    return pairs


def save_parallel(pairs, prefix):
    with open(f'{prefix}.no', 'w', encoding='utf-8') as f_no, \
         open(f'{prefix}.en', 'w', encoding='utf-8') as f_en:
        for nb, en in pairs:
            f_no.write(nb + '\n')
            f_en.write(en + '\n')
    print(f'{prefix}.no / .en  ({len(pairs)} lines)')


def save_tsv(pairs, path):
    with open(path, 'w', encoding='utf-8') as f:
        f.write('norwegian\tenglish\n')
        for nb, en in pairs:
            f.write(f'{nb}\t{en}\n')
    print(f'{path}  ({len(pairs)} pairs)')


def save_jsonl(pairs, path):
    with open(path, 'w', encoding='utf-8') as f:
        for nb, en in pairs:
            json.dump({'source': nb, 'target': en,
                       'source_lang': 'nb', 'target_lang': 'en'},
                      f, ensure_ascii=False)
            f.write('\n')
    print(f'{path}  ({len(pairs)} pairs)')


def save_hf(pairs, path):
    data = {'translation': [{'nb': nb, 'en': en} for nb, en in pairs]}
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    print(f'{path}  ({len(pairs)} pairs)')


_SAVERS = {
    'parallel':    lambda pairs, name: save_parallel(pairs, name),
    'tsv':         lambda pairs, name: save_tsv(pairs, f'{name}.tsv'),
    'jsonl':       lambda pairs, name: save_jsonl(pairs, f'{name}.jsonl'),
    'huggingface': lambda pairs, name: save_hf(pairs, f'{name}_hf.json'),
    'fairseq':     lambda pairs, name: save_parallel(pairs, name),
}


def convert(input_file, fmt='jsonl', output_name=None):
    pairs = parse_tmx(input_file)
    print(f'Extracted {len(pairs)} pairs from {input_file}\n')
    for i, (nb, en) in enumerate(pairs[:3], 1):
        print(f'{i}. {nb}\n   {en}\n')

    name = output_name or input_file.replace('.tmx', '').replace('.', '_')
    saver = _SAVERS.get(fmt)
    if saver is None:
        raise ValueError(f'unknown format: {fmt}')
    saver(pairs, name)


if __name__ == '__main__':
    convert('npd.no.en-nb.tmx', 'jsonl', 'npd_no_en_train')